In [ ]:
import pandas as pd

from sklearn.dummy import DummyRegressor

from sklearn.model_selection import cross_validate,GroupShuffleSplit,GroupKFold

from sklearn.pipeline import make_pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression



In [45]:
path = '../data/processed/clean.csv'

df = pd.read_csv(path)

In [46]:
X = df.drop(columns=['price','id'])
y = df['price']
groups = df['id']

In [47]:
print(X.shape)
print(y.shape)

(21613, 17)
(21613,)


In [48]:
gss = GroupShuffleSplit(test_size=0.2,random_state=42)

In [49]:
train_idx, test_idx = next(gss.split(X, y, groups=groups))

In [50]:
X_train = X.iloc[train_idx]
X_test  = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(17289, 17) (4324, 17)
(17289,) (4324,)


#baseline model

In [51]:

dummy_cross = cross_validate(
    DummyRegressor(), X_train, y_train,
    cv=GroupKFold(n_splits=5),
    groups=groups_train,
    scoring="neg_root_mean_squared_error"
)


print(f'baseline rsme:{-dummy_cross["test_score"].mean()}')

baseline rsme:365047.28943794535


#model 0 - Linear regression

In [52]:

linear_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    LinearRegression()
)

linear_cross = cross_validate(
    linear_pipe, X_train, y_train,
    cv=GroupKFold(n_splits=5),
    groups=groups_train,
    scoring="neg_root_mean_squared_error",
    return_train_score=True
)

train_scores = -linear_cross['train_score']
test_scores = -linear_cross["test_score"]

print(f'train_score:{train_scores.mean()}')
print(f'test_score {test_scores.mean()}')


train_score:201231.68864150112
test_score 201744.7254977328


## Finding: Cv raw price  

| Model | Train RMSE | Validation RMSE | Typical miss vs median (450k) |
|---|---|---|---|
| Baseline (dummy) | - | about 365k | about 81% |
| Linear regression | about 201k | about 201k | about 45% |

Train and validation are close. No overftting. The Linear model beats the baseline.  

#log the target

In [59]:
import numpy as np

In [61]:
print(f'price without log {y_train.skew()}')
print(f'price with log {np.log1p(y_train).skew()}')

price without log 3.997641358024732
price with log 0.4322786020392791


##baseline with logged price

In [63]:
y_train_log = np.log1p(y_train)

dummy_cross_log = cross_validate(
  DummyRegressor(),X_train,y_train_log,
  cv=GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error"
  
)

print(f'baseline  with logged price:{-dummy_cross_log['test_score'].mean()}')

baseline  with logged price:0.5255715434140307


##linear regression with logged price

In [ ]:
linear_pipe_log = make_pipeline(
 SimpleImputer(strategy="median"),
  LinearRegression()
)

linear_cross_log = cross_validate(
  linear_pipe_log,X_train, y_train_log,
  cv = GroupKFold(n_splits=5),
  groups=groups_train,
  scoring="neg_root_mean_squared_error",
  return_train_score=True
  

)


print(f'train_score_log:{-linear_cross_log['train_score'].mean()}')

print(f"test_score_log:{-linear_cross_log['test_score'].mean()}")

train_score_log:0.25334952157753454
test_score_log:0.25368227027130674


## Finding: CV on log price  

| Target | Model | Train RMSE | Validation RMSE |
|---|---|---|---|
| log price | Baseline | - | 0.526 |
| log price | Linear | 0.253 | 0.254 |

Train and validation errors are almost the same so there is no overfitting. Linear halves the baseline error, so it learns real patterns. A typical miss of about 29% (factor 1.29) leaves room to improvement.